# Mamba-Attention Recursive Reasoning Hybrid

This notebook is a smoke-test and command reference for the cycle-level planner and autoregressive task printers. Training logic lives in the typed `scripts.train_*` modules so notebook and CLI behavior cannot diverge.

## 1. Environment

In [ ]:
import torch

device = torch.device(
    "xpu"
    if hasattr(torch, "xpu") and torch.xpu.is_available()
    else ("cuda" if torch.cuda.is_available() else "cpu")
)
print(f"Device: {device}; PyTorch: {torch.__version__}")

## 2. Cycle-Level Planner

`M_min` and `M_max` count complete cycles. Each cycle performs `n_steps` latent updates, one answer-memory update, and one ACT decision.

In [ ]:
from mamba_hybrid.config import MambaHybridConfig
from mamba_hybrid.model import MambaAttentionHybrid

config = MambaHybridConfig(
    d_model=64,
    n_meta=16,
    l_ans=8,
    n_steps=2,
    M_min=1,
    M_max=3,
    vocab_size=32,
    use_cuda_kernels=False,
)
planner = MambaAttentionHybrid(config).to(device)
x_raw = torch.randn(2, 12, config.d_model, device=device)
states, halt_probabilities = planner.forward_state_trajectory(x_raw)
print(
    f"Cycles: {len(states)}; final z: {states[-1][0].shape}; final y: {states[-1][1].shape}"
)

## 3. Autoregressive Printer

The printer sees `[meta memory, answer memory, raw context]` as a bidirectional prefix and generated tokens causally.

In [ ]:
from mamba_hybrid.printer import AutoregressivePrinter

prefix, prefix_mask = planner.build_memory_prefix(x_raw, states[-1])
printer = AutoregressivePrinter(
    config, vocab_size=34, output_vocab_size=32, max_length=8, pad_token_id=33
).to(device)
decoder_inputs = torch.tensor([[32, 1, 2], [32, 3, 4]], device=device)
logits = printer(prefix, decoder_inputs, prefix_mask)
print(f"Teacher-forced logits: {logits.shape}")

## 4. Task-Native Contracts

- **Sudoku:** immutable clues, blank-only CE, unique-solution data, causal digits.
- **Maze:** directional moves with EOS; success means a legal goal-reaching path.
- **Dijkstra:** raw adjacency, explicit source/unreachable token, legal-parent masking, tie-tolerant optimal-tree metrics.
- **GSM8K:** UTF-8 question bytes and normalized final integer only.
- **Multitask:** homogeneous batches routed through one shared MoE planner.

In [ ]:
from importlib import import_module

task_modules = [
    import_module(f"scripts.train_{task}")
    for task in ("dijkstra", "gsm8k", "maze", "multitask", "sudoku")
]
print(f"Imported {len(task_modules)} task-native training modules.")

## 5. Commands

```bash
uv run python -m scripts.download_all_datasets
uv run python -m scripts.train_sudoku
uv run python -m scripts.train_maze
uv run python -m scripts.train_dijkstra
uv run python -m scripts.train_gsm8k
uv run python -m scripts.train_multitask

uv run python -m scripts.evaluate_sudoku
uv run python -m scripts.evaluate_maze
uv run python -m scripts.evaluate_dijkstra
uv run python -m scripts.evaluate_gsm8k
uv run python -m scripts.evaluate_multitask
```

See `TRAINING.md` for target encodings, checkpoint compatibility, metrics, and background-job guidance.